# 🎵 Music Search with CLAP + Weaviate

Build a semantic music search engine that understands natural language queries like:
- "chill lo-fi beats for studying"
- "aggressive metal with fast drums"
- "90s hip hop with jazz samples"

## How it works

1. **Download audio** from YouTube (MusicCaps dataset)
2. **CLAP** embeds the audio waveforms into 512-dimensional vectors
3. **Weaviate** stores the embeddings for fast similarity search
4. Your **text query** gets embedded → finds nearest **audio** embeddings → returns matching songs

This is **true cross-modal search**: text in, audio out!

## Prerequisites

- Google account (you're already here!)
- Weaviate Cloud account (free sandbox): https://console.weaviate.cloud/

---

## Step 1: Setup Environment

First, let's enable GPU and install the required packages.

**Important:** Go to `Runtime → Change runtime type → T4 GPU` before running this notebook!

In [ ]:
# Install required packages (takes ~2-3 minutes)
!pip install -q datasets transformers torch weaviate-client gradio
!pip install -q yt-dlp librosa soundfile

print("✅ Packages installed!")

In [ ]:
# Check if GPU is available
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print("   (CPU will work but be slower)")

## Step 2: Set Up Weaviate Cloud

Get your free Weaviate sandbox:

1. Go to https://console.weaviate.cloud/
2. Sign up / Log in
3. Click **Create Cluster** → Choose **Sandbox** (free)
4. Wait for it to be ready (~1 min)
5. Click on your cluster → **API Keys** → Copy the key
6. Copy the **Cluster URL** (looks like `https://xxx.weaviate.network`)

In [ ]:
# ⬇️ PASTE YOUR WEAVIATE CREDENTIALS HERE ⬇️

WEAVIATE_URL = ""  # e.g., "https://my-sandbox-abc123.weaviate.network"
WEAVIATE_API_KEY = ""  # e.g., "your-api-key-here"

# ⬆️ PASTE YOUR WEAVIATE CREDENTIALS HERE ⬆️

if WEAVIATE_URL and WEAVIATE_API_KEY:
    print("✅ Credentials set!")
else:
    print("❌ Please paste your Weaviate URL and API key above")

## Step 3: Load the CLAP Model

CLAP (from LAION) is like CLIP but for audio:
- **CLIP**: Text ↔ Image (OpenAI)
- **CLAP**: Text ↔ Audio (LAION, open source, FREE)

Both text and audio get embedded into the same 512-dimensional vector space.

In [ ]:
from transformers import ClapModel, ClapProcessor

print("Loading CLAP model (first run downloads ~600MB)...")

model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
model = model.to(device)
model.eval()

print("✅ CLAP model loaded!")

## Step 4: Load MusicCaps Dataset

Google's MusicCaps contains 5,521 music clips with human-written captions:
- 10-second clips from YouTube
- Expert captions describing the music
- Genre/mood labels

We'll download the actual audio from YouTube and embed it with CLAP!

In [ ]:
from datasets import load_dataset

print("Loading MusicCaps dataset...")
dataset = load_dataset("google/MusicCaps", split="train")

print(f"✅ Loaded {len(dataset)} tracks")
print(f"\nColumns: {dataset.column_names}")
print(f"\nExample caption:\n'{dataset[0]['caption']}'")

## Step 5: Download Audio & Generate Embeddings

Now for the real magic! We'll:
1. Download 10-second audio clips from YouTube
2. Embed them using CLAP's **audio encoder**
3. Store the embeddings in Weaviate

This is **true cross-modal search**: text queries find matching audio!

**Note:** Some YouTube videos may be unavailable. We'll skip those and continue.

In [ ]:
import numpy as np
import librosa
import subprocess
import tempfile
import shutil
import os
from tqdm import tqdm

# How many tracks to process (start small - audio download takes time)
LIMIT = 50  # @param {type:"slider", min:10, max:500, step:10}

tracks = []
failed = 0

print(f"Downloading and embedding {LIMIT} audio tracks...\n")
print("⚠️ This takes a while - downloading audio from YouTube\n")

for i in tqdm(range(min(LIMIT, len(dataset)))):
    row = dataset[i]
    ytid = row.get("ytid", "")
    start_s = int(row.get("start_s", 0))
    caption = row.get("caption", "")

    if not ytid:
        continue

    # Use temp directory for clean file handling
    tmp_dir = tempfile.mkdtemp()

    try:
        url = f"https://youtube.com/watch?v={ytid}"
        output_template = os.path.join(tmp_dir, "audio.%(ext)s")

        # Download full audio first
        cmd = [
            "yt-dlp",
            "-x",
            "--audio-format", "wav",
            "-o", output_template,
            "--quiet",
            "--no-warnings",
            url
        ]

        result = subprocess.run(cmd, capture_output=True, timeout=60)

        # Find the downloaded file
        wav_path = os.path.join(tmp_dir, "audio.wav")
        if not os.path.exists(wav_path):
            failed += 1
            continue

        # Load audio with librosa, extracting just the 10-second segment
        # CLAP expects 48kHz sample rate
        audio, sr = librosa.load(
            wav_path,
            sr=48000,
            mono=True,
            offset=start_s,  # Start at the right timestamp
            duration=10      # Load only 10 seconds
        )

        # Embed audio with CLAP
        inputs = processor(audios=[audio], sampling_rate=48000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.get_audio_features(**inputs)

        # Extract embedding tensor
        if hasattr(outputs, 'audio_embeds'):
            embedding = outputs.audio_embeds
        else:
            embedding = outputs

        # Convert to 1D numpy array and normalize
        embedding = embedding.cpu().numpy().flatten()
        embedding = embedding / np.linalg.norm(embedding)

        tracks.append({
            "caption": caption[:500],
            "ytid": ytid,
            "youtube_url": f"https://youtube.com/watch?v={ytid}&t={start_s}",
            "labels": row.get("audioset_positive_labels", ""),
            "embedding": embedding.tolist()
        })

    except Exception as e:
        failed += 1

    finally:
        # Always clean up temp directory
        shutil.rmtree(tmp_dir, ignore_errors=True)

print(f"\n✅ Generated {len(tracks)} audio embeddings")
print(f"   Skipped: {failed} (unavailable videos)")
print(f"   Dimensions: {len(tracks[0]['embedding']) if tracks else 'N/A'}")

## Step 6: Connect to Weaviate & Create Collection

Now we'll store these embeddings in Weaviate for fast similarity search.

In [ ]:
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY)
)

print(f"✅ Connected to Weaviate!")
print(f"   Ready: {client.is_ready()}")

In [ ]:
COLLECTION_NAME = "MusicTrack"

# Delete existing collection if it exists (for clean reruns)
if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)
    print(f"Deleted existing '{COLLECTION_NAME}' collection")

# Create collection with custom vectors (we provide our own CLAP embeddings)
collection = client.collections.create(
    name=COLLECTION_NAME,
    vectorizer_config=Configure.Vectorizer.none(),  # We provide vectors
    properties=[
        Property(name="caption", data_type=DataType.TEXT),
        Property(name="ytid", data_type=DataType.TEXT),
        Property(name="youtube_url", data_type=DataType.TEXT),
        Property(name="labels", data_type=DataType.TEXT),
    ]
)

print(f"✅ Created collection '{COLLECTION_NAME}'")

In [ ]:
# Upload tracks with embeddings
print(f"Uploading {len(tracks)} tracks...")

collection = client.collections.get(COLLECTION_NAME)

# Batch insert for speed
with collection.batch.dynamic() as batch:
    for track in tqdm(tracks):
        batch.add_object(
            properties={
                "caption": track["caption"],
                "ytid": track["ytid"],
                "youtube_url": track["youtube_url"],
                "labels": track["labels"],
            },
            vector=track["embedding"]
        )

# Verify
count = collection.aggregate.over_all(total_count=True).total_count
print(f"\n✅ Uploaded {count} tracks to Weaviate!")

## Step 7: Search UI 🎵

Launch an interactive search interface!

In [ ]:
import gradio as gr

def search_music(query: str, num_results: int = 5):
    """Search for music and return formatted results."""
    if not query.strip():
        return "Please enter a search query."

    # Embed the query with CLAP
    inputs = processor(text=[query], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.get_text_features(**inputs)

    # Extract embedding tensor
    if hasattr(outputs, 'text_embeds'):
        query_embedding = outputs.text_embeds
    elif hasattr(outputs, 'pooler_output'):
        query_embedding = outputs.pooler_output
    else:
        query_embedding = outputs

    # Convert to 1D and normalize
    query_embedding = query_embedding.cpu().numpy().flatten()
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # Search Weaviate
    collection = client.collections.get(COLLECTION_NAME)
    results = collection.query.near_vector(
        near_vector=query_embedding.tolist(),
        limit=num_results,
        return_metadata=["distance"]
    )

    # Format results as HTML
    html = f"<h3>🔍 Results for: \"{query}\"</h3>"

    if not results.objects:
        return html + "<p>No results found.</p>"

    for i, obj in enumerate(results.objects, 1):
        props = obj.properties
        similarity = 1 - (obj.metadata.distance or 0)
        youtube_url = props.get('youtube_url', '')

        html += f"""
        <div style="border: 1px solid #ddd; border-radius: 8px; padding: 12px; margin: 8px 0; background: #f9f9f9;">
            <strong>#{i}</strong> &nbsp; Similarity: <span style="color: green;">{similarity:.2%}</span>
            <p style="margin: 8px 0;">📝 {props.get('caption', 'No caption')}</p>
            <a href="{youtube_url}" target="_blank" style="color: #1a73e8;">🎬 Listen on YouTube</a>
        </div>
        """

    return html

# Create Gradio interface
demo = gr.Interface(
    fn=search_music,
    inputs=[
        gr.Textbox(
            label="Describe the music you're looking for",
            placeholder="e.g., chill lo-fi beats for studying",
            lines=2
        ),
        gr.Slider(
            minimum=1,
            maximum=10,
            value=5,
            step=1,
            label="Number of results"
        )
    ],
    outputs=gr.HTML(label="Results"),
    title="🎵 Music Search",
    description="Search for music using natural language. Powered by CLAP + Weaviate.",
    examples=[
        ["chill lo-fi beats for studying", 5],
        ["aggressive metal with fast drums", 5],
        ["upbeat electronic dance music", 5],
        ["sad piano ballad", 5],
        ["jazz with saxophone", 5],
        ["90s hip hop with jazz samples", 5],
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)  # share=True creates a public link

## Step 8: Cleanup (Optional)

Close the Weaviate connection when done.

In [ ]:
# Close connection when done
client.close()
print("✅ Weaviate connection closed")

---

## 🎉 Congratulations!

You've built a **true cross-modal** music search engine!

### What makes this special

- **Audio embeddings**: We embedded the actual audio waveforms, not just text descriptions
- **Cross-modal search**: Your text query finds matching audio through shared embedding space
- **CLAP magic**: Text and audio live in the same 512-dimensional vector space

### Tech Stack

1. **CLAP** - Open source model that embeds text and audio in the same space (LAION, FREE)
2. **MusicCaps** - Google's dataset of music with captions
3. **yt-dlp** - Downloads audio from YouTube
4. **Weaviate** - Vector database for fast similarity search
5. **Gradio** - Interactive web UI

### Key Concepts

- **Embeddings**: Convert text/audio into numerical vectors that capture meaning
- **Vector similarity**: Similar meanings → vectors close together
- **Cross-modal search**: Search audio using text (or vice versa)
- **Contrastive learning**: How CLAP learns to align text and audio

### Resources

- [CLAP Paper](https://arxiv.org/abs/2211.06687)
- [LAION CLAP on Hugging Face](https://huggingface.co/laion/clap-htsat-unfused)
- [MusicCaps Dataset](https://huggingface.co/datasets/google/MusicCaps)
- [Weaviate Docs](https://weaviate.io/developers/weaviate)
- [Gradio Docs](https://gradio.app/docs/)